# From scratch, with the top solutions' ingredients

Trains the model from random weights with heavy augmentation, CutMix, decoder input masking, a joint CTC loss and weight averaging, at 192 frames. Scored with the same protocol as every other model (3000 random test clips, beam 5, length penalty 0.0) so it can be compared with the 0.307 of the tuned v5 checkpoint. Set SMOKE = False for the real run.

In [ ]:
BRANCH = "phase-5-scratch"
!git clone -q --branch {BRANCH} https://github.com/sohenpatel22/ASL-Fingerspelling-Recognition.git /tmp/repo
%cd /tmp/repo
!pip install -q -e .

In [ ]:
import os, zipfile, json
from pathlib import Path

def find_dir(name, root="/kaggle/input"):
    for base, dirs, _ in os.walk(root):
        if name in dirs:
            return Path(base, name)
        if base[len(root):].count(os.sep) >= 5:
            dirs[:] = []
    raise FileNotFoundError(name)

COMP = next(p for p in (Path("/kaggle/input/competitions/asl-fingerspelling"), Path("/kaggle/input/asl-fingerspelling")) if p.exists())
try:
    NPY_TRAIN, NPY_SUPP = find_dir("npy_train"), find_dir("npy_supp")
except FileNotFoundError:
    for archive in [p for p in Path("/kaggle/input").rglob("*.zip") if "competitions" not in str(p)]:
        with zipfile.ZipFile(archive) as z:
            z.extractall("/tmp/npy")
    NPY_TRAIN, NPY_SUPP = find_dir("npy_train", "/tmp/npy"), find_dir("npy_supp", "/tmp/npy")
DATA = f"data.comp_dir={COMP} data.npy_train={NPY_TRAIN} data.npy_supp={NPY_SUPP}"
import torch
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none", "| cpus:", os.cpu_count())

In [ ]:
SMOKE = True   # about 10 minutes: tiny slices of every stage; set to False for the real run
if SMOKE:
    EXTRA, N_VAL, N_TEST = "train.epochs=2 train.epoch_fraction=0.1 train.eval_samples=200 data.use_supplemental=false", 100, 100
else:
    EXTRA, N_VAL, N_TEST = "", 1000, 3000

!asl-train --config configs/scratch.yaml train.out_dir=/kaggle/working/scratch {DATA} {EXTRA}
assert Path("/kaggle/working/scratch/best.pth").exists(), "training failed: read the output above"
history = [json.loads(l) for l in open("/kaggle/working/scratch/metrics.jsonl")]
for h in history:
    print(int(h["epoch"]), "train", round(h["train_loss"], 3), "val greedy CER", round(h["val_greedy_cer"], 4), "ctc CER", round(h.get("val_ctc_cer", float("nan")), 4), round(h["seconds"]), "s")

In [ ]:
!asl-eval --checkpoint /kaggle/working/scratch/best.pth --config configs/scratch.yaml --split val --max-samples {N_VAL} --sample-seed 0 --length-penalty 0.0 --out /kaggle/working/val_scratch.json --details-out /kaggle/working/val_scratch_details.json {DATA}
!asl-eval --checkpoint /kaggle/working/scratch/best.pth --config configs/scratch.yaml --split test --max-samples {N_TEST} --sample-seed 0 --length-penalty 0.0 --out /kaggle/working/test_scratch.json --details-out /kaggle/working/test_scratch_details.json {DATA}
assert Path("/kaggle/working/test_scratch.json").exists(), "scoring failed: read the output above"
val, test = (json.load(open(f"/kaggle/working/{s}_scratch.json")) for s in ("val", "test"))
print("validation CER:", round(val["cer"], 4), "| test CER:", round(test["cer"], 4), "| corpus CER:", round(test["cer_micro"], 4))
print("for comparison the tuned v5 checkpoint scores 0.2698 on validation and 0.3073 on test (when SMOKE is False)")